In [447]:
import numpy as np
import imutils
import cv2 as cv
import glob 
import pymupdf 
import matplotlib.pyplot as plt
import os
import shutil


In [ ]:
# set up directories 
for d in ['png','errors','upright','with_regions']:
    os.makedirs(f'./data/{d}', exist_ok=True)

### File Conversion

In [ ]:
# printer scans 
for f in ["./data/pdf/01.pdf", "./data/pdf/02.pdf"]:
    doc = pymupdf.open(f)

    # loop and extract pages as pngs 
    for i, page in enumerate(doc):
        pix = page.get_pixmap(dpi=200)
        pix.save(f"./data/png/{f.replace("./data/pdf/", "").replace(".pdf","")}_{i+1}.png")

doc.close()

### Image Reorientation

In [450]:
paths = glob.glob('./data/png/*.png')

# corner ids  
CORNER_IDS = {"top_left": 0, "top_right": 1, "bottom_right": 2, "bottom_left": 3}

# detector and dict
aruco_dict = cv.aruco.getPredefinedDictionary(cv.aruco.DICT_4X4_50)
aruco_params = cv.aruco.DetectorParameters()
detector = cv.aruco.ArucoDetector(aruco_dict, aruco_params)

# regions of image for each q to check (x0, y0, x1, y1)
REGIONS = {
    'tens':     (0.265, 0.185, 0.505, 0.230),
    'ones':     (0.265, 0.232, 0.755, 0.275),
    'q1':       (0.08, 0.340, 0.75, 0.400),
    'q2':       (0.08, 0.460, 0.815, 0.520),
    'q3':       (0.08, 0.580, 0.915, 0.640),
    'q4':       (0.08, 0.690, 0.87, 0.760),
    'target': (0.33, 0.945, 0.49, 0.995),
    'transfer': (0.51, 0.945, 0.67, 0.995),
    'sheet_id': (0.755, 0.020, 0.922, 0.062),
}

# bubbles per region
BUBBLES = {
    'tens': 5, 
    'ones': 10,
    'q1': 3, 
    'q2': 3, 
    'q3': 3, 
    'q4': 5,
    'target': 5, 
    'transfer': 5,
}

In [451]:
# image prep 
def load_image(path):
    # load
    image = cv.imread(path)
    
    if image is None:
        raise FileNotFoundError(f"Could not read {path}")

    # resize, grayscale
    image = imutils.resize(image, width=1000)
    image = cv.cvtColor(image, cv.COLOR_BGR2GRAY)
    
    return image

In [452]:
# grab vertices of img 
def find_corners(image):

    # detect markers
    corners, ids, _ = detector.detectMarkers(image)
    if ids is None:
        raise RuntimeError("No ArUco markers detected.")

    # map each id to its vertices
    markers = {int(mid): c.reshape((4, 2)) for c, mid in zip(corners, ids)}

    missing = [name for name, mid in CORNER_IDS.items() if mid not in markers]
    if missing:
        raise RuntimeError()
    
    # grab outer vertex
    outer = {name: markers[mid][mid] for name, mid in CORNER_IDS.items()}

    return outer["top_left"], outer["bottom_left"], outer["bottom_right"], outer["top_right"]

In [453]:
# reorient img w/ edges 
def transform(image, coords):
    tl, bl, br, tr = coords   

    # set output size based on current dimensions 
    width = int(max(np.linalg.norm(tl - tr), np.linalg.norm(bl - br)))
    height = int(max(np.linalg.norm(tl - bl), np.linalg.norm(tr - br)))

    # original and new vertices 
    input_pts = np.float32([tl, bl, br, tr])
    output_pts = np.float32([[0, 0], [0, height - 1],
                             [width - 1, height - 1], [width - 1, 0]])

    # transform
    M = cv.getPerspectiveTransform(input_pts, output_pts)
    corrected = cv.warpPerspective(image, M, (width, height), flags=cv.INTER_LINEAR)

    # resize to original width
    return imutils.resize(corrected, width=1000)

In [454]:
# prep image 
def prepare(path):
    image = load_image(path)
    corners = find_corners(image)
    return transform(image, corners)

In [455]:
# export upright images; failures go to errors
error_paths = []
for path in paths:
    name = os.path.splitext(os.path.basename(path))[0]

    # try to write image into upright ; if error (incl aruco not found), 
    try:
        cv.imwrite(f'./data/upright/{name}.png', prepare(path))

    except (FileNotFoundError, RuntimeError, cv.error) as e:
        dest_path = os.path.join('./data/errors', os.path.basename(path))

        if os.path.exists(path):
            shutil.move(path, dest_path)   
            
        error_paths.append(path)

# refresh the list so later cells skip the moved files
paths = glob.glob('./data/png/*.png')
print(f"{len(error_paths)} file(s) moved to ./data/errors")

1 file(s) moved to ./data/errors


### Regions of Interest

In [456]:
# convert fractions for regions to pixels by scaling width and height 
def to_px(frac, shape):
    H, W = shape

    # scale sizes 
    return (int(frac[0] * W), int(frac[1] * H),
            int(frac[2] * W), int(frac[3] * H))

In [457]:
# show sheet with regions 
def plot_regions(image):

    fig, ax = plt.subplots(figsize=(12, 9))
    ax.imshow(image, cmap='gray')

    # regions in green
    for frac in REGIONS.values():
        x0, y0, x1, y1 = to_px(frac, image.shape)
        ax.add_patch(plt.Rectangle((x0, y0), x1 - x0, y1 - y0, color='lime', fill=False))

    ax.set_axis_off()
    plt.show()

In [458]:
for path in glob.glob('./data/upright/*.png'):
    # load 
    image = load_image(path)

    # color copy so the boxes show up green
    display_image = cv.cvtColor(image, cv.COLOR_GRAY2BGR)

    for _, frac in REGIONS.items():
        # prepare region
        x0, y0, x1, y1 = to_px(frac, image.shape)

        # draw region onto display
        cv.rectangle(display_image, (x0, y0), (x1, y1), (0, 255, 0), 2)

    name = os.path.splitext(os.path.basename(path))[0]
    cv.imwrite(f'./data/with_regions/{name}.png', display_image)
